In [1]:
import sys
from pathlib import Path
project_root = Path.cwd().parent    # notebooks -> project_root
sys.path.insert(0, str(project_root))
print(project_root)

from matplotlib import pyplot as plt
import numpy as np

/Users/tcui0786/Work/DeepTransport/PyFInS-public


In [2]:
from NumpyModels.RandomModel import RandomModel
from NumpyModels import GaussianDiagonalPrior
from BaseModels import GaussianLikelihood
from NumpyModels.DerivativeTests import test_combined

m = 30
n = 100
sig2noise = 5.0

prior = GaussianDiagonalPrior(dim=n, mu=0.0, decay_rate=2.0, variance=4.0)

# True parameter
true_u = prior.sample(1)
model = RandomModel(m, n, nonlinear=True)
y, _ = model.solve(true_u)

sigma = np.abs(y).mean()/sig2noise
fake_data = y + np.random.normal(0, sigma, size=y.shape)
like = GaussianLikelihood(fake_data, sigma)

test_combined(prior, model, n_tests=50)

Solver and prior derivative tests
  number of tests : 50
  epsilon         : 1.000e-06

test                             mean relative error    max relative error
--------------------------------------------------------------------------
prior L/L.T adjoint                     1.901474e-16          2.245315e-15
prior inverse                           4.249860e-17          7.368125e-17
physical Jacobian                       2.464692e-09          3.389863e-08
physical adjoint                        7.076891e-16          4.744199e-15
whitened Jacobian                       4.604896e-09          1.536947e-08
whitened adjoint                        8.031831e-16          8.991881e-15


In [5]:
print(sigma)

0.011949674765414243


In [ ]:
from NumpyInference import LikelihoodInformedSMC, LikelihoodInformedSubspace
from InferenceTools.TransitionKernels import OneStepKernel

kernel = OneStepKernel(
        type="OW",
        sigma=-3.0,
        adapt=True,
        nbatch=1,
)

LIS = LikelihoodInformedSubspace(epsilon=0.1)

sampler = LikelihoodInformedSMC(
        prior,
        model,
        like,
        LIS,
        kernel,
        max_iter=100,
        n_samples=1000,
        n_mutation_steps=20,
        n_final_mutation_steps=100, # n_samples times the final mutation steps give the total number of samples 
        ess_tol=0.5,
        min_beta=1e-6,
        pm_sample_size=5,
)

sampler.iterate()

iteration 0
Start model evaluations


100%|██████████| 1000/1000 [00:00<00:00, 78727.83it/s]


Total time for model evaluations is 0.05425187497166917
temperature 0.028167791972929034, ess 0.489200068471693, log const -1.9793604132696834
LIS dimension 5


MCMC: 100%|██████████| 20/20 [00:00<00:00, 21.54it/s, acc_rate=0.788, sigma=-2.00000]


iteration 1
Start model evaluations



100%|██████████| 1000/1000 [00:00<00:00, 70204.61it/s]


... (10 outputs left out)


iteration 4
Start model evaluations



100%|██████████| 1000/1000 [00:00<00:00, 58933.60it/s]


Total time for model evaluations is 0.01786208403063938
temperature 1.0, ess 0.5091963263626988, log const -8.35445124744106
LIS dimension 18


MCMC: 100%|██████████| 100/100 [00:04<00:00, 21.35it/s, acc_rate=0.485, sigma=-2.10000]
